Construction of ESG scores

TF/IDF as main measure, mean TF/IDF score as very good variable

Work done on E/G primary indicator words

How should S be introduced? It can't be used to differentiate. But a consistently higher ESG score indicates social stewardship, which can be both E or G. So therefore if a company does well in year N, Year N+1 gets a slight positive bonus, and vice versa, because a decrease is a failure of stewardship

Social does deserve a smaller weighting however, because when viewed relatively, firms have very similar social discourse intensity, but by industry there are slight differentiations.

Normalized search volume for that year is another variable. A high environmental discourse score in a year with high environmental attention could be optics or genuine concern. Nevertheless, it should be a positive variable.



In [ ]:
import math
import polars as pl
import yaml

# Paths
PARQUET_FILE = "spy_10k_2015_present.parquet"
LEXICON_FILE = "ESG_Lexicon.yml"
TRENDS_FILE = "google_trends_pillars.csv"  # year,E,S,G

START_YEAR = 2015
END_YEAR = 2024

# Robust weighting controls
SHRINK_K = 0.35          # how much we trust Google Trends vs equal weights (0 = equal, 1 = fully trends)
STEW_W = 0.30            # stewardship precedence (30% of final raw score)
DF_CAP_SHARE = 0.98      # drop ultra-common terms
CHUNK_SIZE = 12          # lower RAM

# Load lexicon
with open(LEXICON_FILE, "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

pillars = {
    "E": list(lex.get("environmental") or []),
    "S": list(lex.get("social") or []),
    "G": list(lex.get("governance") or []),
}

# Flatten terms as (pillar, regex)
terms = []
for p, pats in pillars.items():
    for pat in pats:
        terms.append((p, pat))

# Load firm-year docs
df = pl.read_parquet(PARQUET_FILE, columns=["cik", "gics_sector", "filing_period", "text"])

docs = (
    df.with_columns(
        pl.col("cik").cast(pl.Utf8),
        pl.col("gics_sector").cast(pl.Utf8),
        pl.col("filing_period").cast(pl.Date),
        pl.col("text").cast(pl.Utf8).fill_null(""),
        pl.col("filing_period").cast(pl.Date).dt.year().alias("year"),
    )
    .filter(
        pl.col("cik").is_not_null()
        & (pl.col("year") >= START_YEAR)
        & (pl.col("year") <= END_YEAR)
    )
    .group_by(["cik", "gics_sector", "year"])
    .agg(pl.col("text").str.concat("\n").alias("text"))
    .with_columns(pl.col("text").str.count_matches(r"(?i)\b\w+\b").alias("n_words"))
    .filter(pl.col("n_words") > 0)
)

N = int(docs.select(pl.len()).item())

# Compute pillar TF-IDF per firm-year (global IDF)
docs_work = docs.select(["cik", "gics_sector", "year", "text", "n_words"]).with_columns(
    pl.lit(0.0).alias("tfidf_E"),
    pl.lit(0.0).alias("tfidf_S"),
    pl.lit(0.0).alias("tfidf_G"),
)

pillar_to_col = {"E": "tfidf_E", "S": "tfidf_S", "G": "tfidf_G"}

for start in range(0, len(terms), CHUNK_SIZE):
    chunk = terms[start:start + CHUNK_SIZE]
    cnt_cols = [f"c{start+i:03d}" for i in range(len(chunk))]

    tmp = docs_work.with_columns(
        [pl.col("text").str.count_matches(pat).alias(col) for (p, pat), col in zip(chunk, cnt_cols)]
    )

    idf_vals = []
    for col in cnt_cols:
        df_i = int(tmp.select((pl.col(col) > 0).sum()).item())
        if df_i / N >= DF_CAP_SHARE:
            idf_vals.append(0.0)
        else:
            idf_vals.append(math.log((N + 1) / (df_i + 1)) + 1.0)

    expr_updates = {}
    for (p, _), col, idf in zip(chunk, cnt_cols, idf_vals):
        if idf == 0.0:
            continue
        tfidf_term = (pl.col(col).cast(pl.Float64) / pl.col("n_words").cast(pl.Float64)) * pl.lit(idf)
        target = pillar_to_col[p]
        expr_updates[target] = expr_updates.get(target, pl.lit(0.0)) + tfidf_term

    if expr_updates:
        tmp = tmp.with_columns([(pl.col(k) + v).alias(k) for k, v in expr_updates.items()])

    docs_work = tmp.drop(cnt_cols)

firm_year = docs_work.select(["cik", "gics_sector", "year", "tfidf_E", "tfidf_S", "tfidf_G"])

# Year-standardise (z-score within year)
firm_year = firm_year.join(
    firm_year.group_by("year").agg(
        pl.mean("tfidf_E").alias("muE"), pl.std("tfidf_E").alias("sdE"),
        pl.mean("tfidf_S").alias("muS"), pl.std("tfidf_S").alias("sdS"),
        pl.mean("tfidf_G").alias("muG"), pl.std("tfidf_G").alias("sdG"),
    ),
    on="year",
    how="left",
).with_columns(
    ((pl.col("tfidf_E") - pl.col("muE")) / pl.col("sdE")).fill_nan(0.0).fill_null(0.0).alias("zE"),
    ((pl.col("tfidf_S") - pl.col("muS")) / pl.col("sdS")).fill_nan(0.0).fill_null(0.0).alias("zS"),
    ((pl.col("tfidf_G") - pl.col("muG")) / pl.col("sdG")).fill_nan(0.0).fill_null(0.0).alias("zG"),
).drop(["muE","sdE","muS","sdS","muG","sdG"])

# Load Google trends and build robust weights
tr = pl.read_csv(TRENDS_FILE).with_columns(pl.col("year").cast(pl.Int32))
tr = tr.with_columns(
    (pl.col("E") + pl.col("S") + pl.col("G")).alias("tot")
).with_columns(
    (pl.col("E") / pl.col("tot")).alias("shareE"),
    (pl.col("S") / pl.col("tot")).alias("shareS"),
    (pl.col("G") / pl.col("tot")).alias("shareG"),
).with_columns(
    # shrink toward equal weights so we don't get insane year-to-year swings
    ((1 - SHRINK_K) * (1/3) + SHRINK_K * pl.col("shareE")).alias("wE"),
    ((1 - SHRINK_K) * (1/3) + SHRINK_K * pl.col("shareS")).alias("wS"),
    ((1 - SHRINK_K) * (1/3) + SHRINK_K * pl.col("shareG")).alias("wG"),
).select(["year", "wE", "wS", "wG"])

firm_year = firm_year.join(tr, on="year", how="left").with_columns(
    # fallback if trends missing for a year
    pl.col("wE").fill_null(1/3),
    pl.col("wS").fill_null(1/3),
    pl.col("wG").fill_null(1/3),
)

# Base ESG (weighted pillars)
firm_year = firm_year.with_columns(
    (pl.col("wE") * pl.col("zE") + pl.col("wS") * pl.col("zS") + pl.col("wG") * pl.col("zG")).alias("base_esg")
)

# Stewardship precedence: reward improvement vs last year
firm_year = firm_year.sort(["cik", "year"]).with_columns(
    (pl.col("base_esg") - pl.col("base_esg").shift(1)).over("cik").fill_null(0.0).alias("stewardship")
)

# Final raw score: stewardship gets a fixed precedence share
firm_year = firm_year.with_columns(
    (((1 - STEW_W) * pl.col("base_esg")) + (STEW_W * pl.col("stewardship"))).alias("esg_raw")
)

# Map to 0–100 by year percentile (robust, easy to explain)
firm_year = firm_year.with_columns(
    (pl.col("esg_raw").rank(method="average").over("year") / pl.len().over("year") * 100.0).alias("esg_0_100")
)

# Output table for SQL write-back or plotting
out = firm_year.select(["cik", "gics_sector", "year", "tfidf_E", "tfidf_S", "tfidf_G", "base_esg", "stewardship", "esg_0_100"])
out